Fetches user-sourced data from MongoDB and uses it to retrain the ML classifier

In [17]:
import pandas as pd
from PIL import Image
import numpy as np

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import os
from dotenv import load_dotenv
from pymongo import MongoClient

import base64
import io

In [18]:
load_dotenv("../src/credentials/.env")
db_url = os.getenv("MONGO_CONNECTION_STRING")

client = MongoClient(db_url)
db = client["classifier"]
collection = db["feedback"]

In [19]:
# Exploring the user-sourced dataset

total = collection.count_documents({})
num_correct = collection.count_documents({"was_correct" : True})
num_incorrect = collection.count_documents({"was_correct" : False})

print(f"Out of {total} responses, {num_correct} were correct, and {num_incorrect} were incorrect")

classes = collection.distinct("true_label")
print(f"Received samples from {len(classes)} different classes")

worst = []
better = []
perfect = []

for char in classes:
    num_char = collection.count_documents({"true_label" : char})
    num_correct = collection.count_documents({
        "true_label" : char,
        "was_correct" : True
    })
    if (num_correct <= (num_char/2)):
        worst.append(char)
    elif (num_correct == num_char):
        perfect.append(char)
    else:
        better.append(char)
    print(f"\"{char}\": {num_correct}/{num_char} = {((num_correct/num_char)*100):.2f}% correct predictions")

print(f"The hardest characters to predict: {worst}")
print(f"The model was better at predicting: {better}")
print(f"Characters that were perfectly predicted: {perfect}")

Out of 930 responses, 526 were correct, and 404 were incorrect
Received samples from 62 different classes
"0": 4/15 = 26.67% correct predictions
"1": 0/15 = 0.00% correct predictions
"2": 6/15 = 40.00% correct predictions
"3": 8/15 = 53.33% correct predictions
"4": 4/15 = 26.67% correct predictions
"5": 11/15 = 73.33% correct predictions
"6": 13/15 = 86.67% correct predictions
"7": 15/15 = 100.00% correct predictions
"8": 2/15 = 13.33% correct predictions
"9": 15/15 = 100.00% correct predictions
"A": 14/15 = 93.33% correct predictions
"B": 15/15 = 100.00% correct predictions
"C": 13/15 = 86.67% correct predictions
"D": 9/15 = 60.00% correct predictions
"E": 15/15 = 100.00% correct predictions
"F": 15/15 = 100.00% correct predictions
"G": 13/15 = 86.67% correct predictions
"H": 14/15 = 93.33% correct predictions
"I": 0/15 = 0.00% correct predictions
"J": 4/15 = 26.67% correct predictions
"K": 12/15 = 80.00% correct predictions
"L": 14/15 = 93.33% correct predictions
"M": 7/15 = 46.67% c

In [20]:
# Make a list of all user-drawn images
resized_img = []
y = []

data = collection.find({})
for entry in data:
    imageData = entry["image_b64"]
    bytes = base64.b64decode(imageData.split(",")[1])

    img = Image.open(io.BytesIO(bytes))
    img = img.resize((64, 64))
    
    resized_img.append(img)
    y.append(entry["true_label"])

In [21]:
# Combine with previous dataset
# Extract the sample paths and labels
csv = pd.read_csv('../data/english.csv')
paths = csv['image'].tolist()
y.extend(csv['label'].tolist())

# Get the 62 classes
classes = csv['label'].unique().tolist()

# Convert class labels to integer
conversion = {}

cur = 0
for label in classes:
    conversion[label] = cur
    cur += 1

for i in range(len(y)):
    y[i] = conversion[f"{y[i]}"]

for image in paths:
    img = Image.open(f"../data/{image}")
    img = img.resize((64, 64))
    
    resized_img.append(img)

In [22]:
# Preprocess and stack
X = []
for img in resized_img:
    img = img.convert("L")

    img_arr = np.array(img)
    img_arr = img_arr / 255.0

    img_arr = np.expand_dims(img_arr, axis=-1)

    X.append(img_arr)

X = np.array(X, dtype="float32")
y = np.array(y, dtype="int32")

In [23]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
# Try 1NN first
X_flat = X.reshape(X.shape[0], -1)

X_flat_train, X_flat_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

def one_nn(X_flat_train, y_train, X_flat_test):
    y_pred = []
    for test in X_flat_test:
        dist = np.sum((X_flat_train - test) ** 2, axis=1)
        y_pred.append(y_train[np.argmin(dist)])
    
    return np.array(y_pred)

y_pred = one_nn(X_flat_train, y_train, X_flat_test)

# Evaluate
print(classification_report(y_test, y_pred, target_names=classes))

              precision    recall  f1-score   support

           0       0.20      0.57      0.30         7
           1       0.30      0.50      0.37        16
           2       0.60      0.50      0.55        18
           3       0.50      0.64      0.56        14
           4       0.54      0.50      0.52        14
           5       0.40      0.15      0.22        13
           6       0.50      0.43      0.46        14
           7       0.73      0.92      0.81        12
           8       0.70      0.47      0.56        15
           9       0.58      0.61      0.59        18
           A       0.88      0.74      0.80        19
           B       0.69      0.52      0.59        21
           C       0.56      0.67      0.61        15
           D       0.70      0.58      0.64        12
           E       0.56      0.56      0.56        16
           F       0.44      0.36      0.40        11
           G       0.62      0.36      0.45        14
           H       0.36    

In [25]:
# Create CNN
def init_model():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(62, activation="softmax"))

    return model

In [26]:
model = init_model()
model.summary()

C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 690,622 (2.63 MB)

 Trainable params: 690,622 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_test, y_test))

Epoch 1/10


C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.0510 - loss: 3.9276 - val_accuracy: 0.2742 - val_loss: 3.0236
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3344 - loss: 2.4893 - val_accuracy: 0.6014 - val_loss: 1.4956
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5579 - loss: 1.5688 - val_accuracy: 0.7316 - val_loss: 1.0144
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6503 - loss: 1.1471 - val_accuracy: 0.7339 - val_loss: 0.9069
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7339 - loss: 0.8737 - val_accuracy: 0.7846 - val_loss: 0.7681
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7759 - loss: 0.7151 - val_accuracy: 0.7961 - val_loss: 0.7035
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8168 - loss: 0.5656 - val_accuracy: 0.8041 - val_loss: 0.6794
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8413 - loss: 0.4822 - val_accuracy: 0.793

In [28]:
# Evaluation
print(model.evaluate(X_test, y_test))

y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=classes))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7938 - loss: 0.7869
[0.7869025468826294, 0.7937787771224976]
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

           0       0.40      0.57      0.47         7
           1       0.77      0.62      0.69        16
           2       0.65      0.83      0.73        18
           3       0.76      0.93      0.84        14
           4       0.74      1.00      0.85        14
           5       0.91      0.77      0.83        13
           6       0.75      0.86      0.80        14
           7       1.00      1.00      1.00        12
           8       0.48      0.93      0.64        15
           9       1.00      0.61      0.76        18
           A       0.82      0.95      0.88        19
           B       0.94      0.81      0.87        21
           C       0.69      0.60      0.64        15
           D       1.00      1.00      1.00        12
           E       1.00      0.94   

In [29]:
# Exporting model
model.save("../src/models/char_cnn_feedback.keras")

Recall that we saw between 71-75% accuracy with the model on the original dataset.
Here we see 79% accuracy. The addition of the user-generated samples was able to increase the accuracy of the model.